# 05 — Ablation Study, Error Analysis & Final Held-Out Test Evaluation

**Pipeline Stage:** Final evaluation (run after notebooks 01–04)

**Outputs:**
- `artifacts/results/ablation_summary.csv`
- `artifacts/results/per_class_metrics.csv`
- `artifacts/results/confusion_matrices.png`

> ⚠️ **This notebook performs a one-time final evaluation on the held-out test set.**  
> Do not re-run repeatedly or tune hyperparameters based on these results.

---
## Section 1 — Imports, Config, Calibration Application Helper

In [ ]:
import os
import json
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax
from scipy.stats import entropy

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
@dataclass
class CFG:
    SEEDS: List[int]             = field(default_factory=lambda: [42, 2026, 777])
    MODEL_NAME_SMALL: str        = 'Qwen/Qwen3-4B'
    MODEL_NAME_MAIN: str         = 'Qwen/Qwen3-8B'
    DATA_DIR: str                = r'c:\Users\Public\Documents\TA\Dataset'
    ARTIFACT_DIR: str            = r'c:\Users\Public\Documents\TA\Dataset\artifacts'
    N_FOLDS: int                 = 5
    MAX_HISTORY_TURNS: int       = 30
    MAX_LENGTH: int              = 1536
    # label definitions
    CLASS_NAMES: List[str]       = field(default_factory=lambda: [
        'C0-NoDef', 'C1-Action', 'C2-MajorImgDist', 'C3-Disavowal',
        'C4-MinorImgDist', 'C5-Neurotic', 'C6-Obsessional', 'C7-HighAdapt', 'C8-NMI'
    ])

cfg = CFG()

# Seed everything
random.seed(cfg.SEEDS[0])
np.random.seed(cfg.SEEDS[0])

# Paths
DATA_DIR   = Path(cfg.DATA_DIR)
ART_DIR    = Path(cfg.ARTIFACT_DIR)
RESULTS    = ART_DIR / 'results'
SPLITS     = ART_DIR / 'splits'
STAGE1     = ART_DIR / 'stage1_qwen3_8b' / 'seed_42'
STAGE2     = ART_DIR / 'stage2_sequential'

RESULTS.mkdir(parents=True, exist_ok=True)
print('Config OK — artifact root:', ART_DIR)

In [ ]:
# ---------------------------------------------------------------------------
# Calibration helper — mirrors the methods from notebook 04 & PRD specs
# ---------------------------------------------------------------------------
def apply_calibration(
    logits: np.ndarray,
    method: str,
    params: dict,
    prior_probs: Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    Apply structured calibration to raw logits matching Notebook 04 & PRD specs.
    Returns probabilities, shape (N, 9).
    """
    z = logits.copy().astype(float)
    m = method.upper() if isinstance(method, str) else 'C'

    if m in ['C', 'METHOD_C']:
        # Method C: Structured C0/C7 Gate + Prior Adjustment
        b0 = params.get('b0', 0.0)
        b7 = params.get('b7', 0.0)
        tau = params.get('tau', 0.5)
        log_pi = np.log((prior_probs if prior_probs is not None else np.ones(9)/9.0) + 1e-12)
        z[:, 0] = z[:, 0] + b0
        z[:, 7] = z[:, 7] + b7
        for c in list(range(1, 7)) + [8]:
            z[:, c] = z[:, c] - tau * log_pi[c]

    elif m in ['A', 'METHOD_A']:
        # Method A: Prior Logit Adjustment
        tau = params.get('tau', 0.5)
        log_pi = np.log((prior_probs if prior_probs is not None else np.ones(9)/9.0) + 1e-12)
        z = z - tau * log_pi[np.newaxis, :]

    elif m in ['B', 'METHOD_B']:
        # Method B: Temperature Scaling + Prior Adjustment
        T = params.get('T', 1.0)
        tau = params.get('tau', 0.5)
        log_pi = np.log((prior_probs if prior_probs is not None else np.ones(9)/9.0) + 1e-12)
        z = z / T - tau * log_pi[np.newaxis, :]

    elif m in ['RANDOM', 'RANDOM_BIAS']:
        delta = np.array(params.get('delta', [0.0]*9))
        z = z + delta[np.newaxis, :]

    return softmax(z, axis=1)

print('apply_calibration() defined (PRD-compliant)')


---
## Section 2 — Load Data & Build Ablation Matrix

In [ ]:
# ---------------------------------------------------------------------------
# 2.1  Load fold assignment
# ---------------------------------------------------------------------------
fold_df = pd.read_parquet(SPLITS / 'train_folds.parquet')
print('fold_df shape:', fold_df.shape)
print(fold_df.head(3))
print('Columns:', fold_df.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# 2.2  Load train & test JSON
# ---------------------------------------------------------------------------
with open(DATA_DIR / 'train.json') as f:
    train_data = json.load(f)
with open(DATA_DIR / 'test.json') as f:
    test_data = json.load(f)

print(f'Train samples: {len(train_data)}  Test samples: {len(test_data)}')

train_labels = np.array([s['label'] for s in train_data])
test_labels  = np.array([s['label'] for s in test_data])
train_ids    = [s['id'] for s in train_data]
test_ids     = [s['id'] for s in test_data]
test_dialogue_ids = [s['dialogue_id'] for s in test_data]

print('Train label distribution:', dict(zip(*np.unique(train_labels, return_counts=True))))
print('Test  label distribution:', dict(zip(*np.unique(test_labels,  return_counts=True))))

In [ ]:
# ---------------------------------------------------------------------------
# 2.3  Load Stage 1 logits
# ---------------------------------------------------------------------------
s1_oof_path  = STAGE1 / 'oof_logits.npy'
s1_test_path = STAGE1 / 'test_logits_by_fold.npy'

assert s1_oof_path.exists(),  f'Missing: {s1_oof_path}'
assert s1_test_path.exists(), f'Missing: {s1_test_path}'

s1_oof_logits        = np.load(s1_oof_path)   # (1864, 9)
s1_test_by_fold      = np.load(s1_test_path)  # (5, 472, 9)
print('Stage1 OOF logits:', s1_oof_logits.shape)
print('Stage1 test-by-fold:', s1_test_by_fold.shape)

In [ ]:
# ---------------------------------------------------------------------------
# 2.4  Load Stage 2 logits
# ---------------------------------------------------------------------------
s2_oof_lgbm_path   = STAGE2 / 'oof_logits_lgbm.npy'
s2_oof_cat_path    = STAGE2 / 'oof_logits_catboost.npy'
s2_test_lgbm_path  = STAGE2 / 'test_logits_routed_lgbm.npy'
s2_test_cat_path   = STAGE2 / 'test_logits_routed_catboost.npy'

assert s2_oof_lgbm_path.exists(),  f'Missing: {s2_oof_lgbm_path}'
assert s2_oof_cat_path.exists(),   f'Missing: {s2_oof_cat_path}'
assert s2_test_lgbm_path.exists(), f'Missing: {s2_test_lgbm_path}'
assert s2_test_cat_path.exists(),  f'Missing: {s2_test_cat_path}'

s2_oof_lgbm  = np.load(s2_oof_lgbm_path)   # (1864, 9)
s2_oof_cat   = np.load(s2_oof_cat_path)    # (1864, 9)
s2_test_lgbm = np.load(s2_test_lgbm_path)  # (472, 9)
s2_test_cat  = np.load(s2_test_cat_path)   # (472, 9)

print('Stage2 OOF LGBM:', s2_oof_lgbm.shape)
print('Stage2 OOF CatBoost:', s2_oof_cat.shape)
print('Stage2 test LGBM routed:', s2_test_lgbm.shape)
print('Stage2 test CatBoost routed:', s2_test_cat.shape)

In [ ]:
# ---------------------------------------------------------------------------
# 2.5  Load calibration params
# ---------------------------------------------------------------------------
cal_params_path = ART_DIR / 'results' / 'best_calibration_params.json'
assert cal_params_path.exists(), f'Missing: {cal_params_path}'

with open(cal_params_path) as f:
    best_cal_params = json.load(f)

print('Best calibration params:', json.dumps(best_cal_params, indent=2))

best_method = best_cal_params.get('best_method', best_cal_params.get('method', 'C'))
if 'params' in best_cal_params:
    best_params = best_cal_params['params']
elif 'method_c' in best_cal_params:
    best_params = best_cal_params['method_c']
    best_method = 'C'
else:
    best_params = {}

print(f'Using calibration method: {best_method!r}')
print('Parameters:', best_params)

# Class prior probabilities (from full train set)
train_label_counts = np.bincount(train_labels, minlength=9)
prior_probs = train_label_counts / train_label_counts.sum()
print('Train prior probs:', np.round(prior_probs, 4))


In [ ]:
# ---------------------------------------------------------------------------
# 2.6  Held-out test routing for Stage 1
# ---------------------------------------------------------------------------
def route_test_logits(
    test_data: list,
    fold_df: pd.DataFrame,
    test_logits_by_fold: np.ndarray,  # (5, N_test, 9)
) -> np.ndarray:
    """
    For each test sample, pick the fold where its dialogue_id was held-out
    during training (i.e., the fold where that dialogue_id appears as *val*).

    For dialogue_ids absent from the train set entirely, use the mean over
    all 5 folds (ensemble).

    Returns routed logits of shape (N_test, 9).
    """
    n_folds = test_logits_by_fold.shape[0]
    n_test  = len(test_data)

    # Build mapping: dialogue_id -> fold where it was held-out
    # In a 5-fold CV, a dialogue_id appears in validation exactly when
    # fold_id == that fold.  fold_df['fold_id'] is the TRAIN fold index (0-4)
    # meaning the sample was used for validation on fold fold_id.
    # (consistent with standard StratifiedGroupKFold conventions used in nb01)
    dlg_to_heldout_fold: Dict[str, int] = {}
    for dlg_id, grp in fold_df.groupby('dialogue_id'):
        # All rows of a dialogue share the same fold_id (GroupKFold guarantee)
        fold_ids = grp['fold_id'].unique()
        if len(fold_ids) == 1:
            dlg_to_heldout_fold[str(dlg_id)] = int(fold_ids[0])
        else:
            # Shouldn't happen, but fall back to most common
            dlg_to_heldout_fold[str(dlg_id)] = int(grp['fold_id'].mode()[0])

    mean_logits = test_logits_by_fold.mean(axis=0)  # (N_test, 9)

    routed = np.empty((n_test, test_logits_by_fold.shape[2]))
    unseen_count = 0
    for i, sample in enumerate(test_data):
        dlg_id = str(sample['dialogue_id'])
        if dlg_id in dlg_to_heldout_fold:
            fold_idx = dlg_to_heldout_fold[dlg_id]
            routed[i] = test_logits_by_fold[fold_idx, i]
        else:
            routed[i] = mean_logits[i]
            unseen_count += 1

    print(f'route_test_logits: {n_test} test samples, {unseen_count} used mean-fold fallback')
    return routed


s1_test_routed = route_test_logits(test_data, fold_df, s1_test_by_fold)
s1_test_mean   = s1_test_by_fold.mean(axis=0)   # non-routed (for inflation measurement)
print('Stage1 test routed logits shape:', s1_test_routed.shape)
print('Stage1 test mean   logits shape:', s1_test_mean.shape)

In [ ]:
# ---------------------------------------------------------------------------
# 2.7  Define 4 ablation conditions
# ---------------------------------------------------------------------------
# Each condition is a dict with:
#   name        : str
#   oof_logits  : (1864, 9) array
#   test_logits : (472, 9) array   <- held-out-routed where applicable
#   calibrate   : bool

conditions = [
    {
        'name'        : 'Baseline',
        'oof_logits'  : s1_oof_logits,
        'test_logits' : s1_test_routed,
        'calibrate'   : False,
    },
    {
        'name'        : 'Baseline + Structured Calibration (RQ2)',
        'oof_logits'  : s1_oof_logits,
        'test_logits' : s1_test_routed,
        'calibrate'   : True,
    },
    {
        'name'        : 'Baseline + Sequential Stage (RQ1)',
        'oof_logits'  : s2_oof_lgbm,
        'test_logits' : s2_test_lgbm,
        'calibrate'   : False,
    },
    {
        'name'        : 'Full Pipeline (RQ1 + RQ2)',
        'oof_logits'  : s2_oof_lgbm,
        'test_logits' : s2_test_lgbm,
        'calibrate'   : True,
    },
]

# Pre-compute calibrated logit arrays for each condition
for cond in conditions:
    if cond['calibrate']:
        cond['oof_probs']  = apply_calibration(cond['oof_logits'],  best_method, best_params, prior_probs)
        cond['test_probs'] = apply_calibration(cond['test_logits'], best_method, best_params, prior_probs)
    else:
        cond['oof_probs']  = softmax(cond['oof_logits'],  axis=1)
        cond['test_probs'] = softmax(cond['test_logits'], axis=1)
    cond['oof_preds']  = cond['oof_probs'].argmax(axis=1)
    cond['test_preds'] = cond['test_probs'].argmax(axis=1)

print('Ablation conditions defined:')
for c in conditions:
    print(f"  [{c['name']}]  OOF preds shape: {c['oof_preds'].shape}")

---
## Section 3 — Zero-Leakage & Causal Integrity Unit Tests

In [ ]:
# ---------------------------------------------------------------------------
# 3.1  Zero-leakage test — all 5 outer folds
# ---------------------------------------------------------------------------
print('=' * 60)
print('ZERO-LEAKAGE UNIT TESTS — OUTER FOLDS')
print('=' * 60)

leakage_results = []
for fold_id in range(cfg.N_FOLDS):
    val_mask  = fold_df['fold_id'] == fold_id
    train_mask = ~val_mask

    val_dialogues   = set(fold_df.loc[val_mask,   'dialogue_id'].unique())
    train_dialogues = set(fold_df.loc[train_mask, 'dialogue_id'].unique())

    overlap = train_dialogues & val_dialogues
    n_viol  = len(overlap)

    if n_viol == 0:
        status = 'PASS'
        print(f'  Fold {fold_id}: ZERO LEAKAGE: PASS  '
              f'(train_dlg={len(train_dialogues)}, val_dlg={len(val_dialogues)})')
    else:
        status = f'FAIL ({n_viol} violations)'
        print(f'  Fold {fold_id}: ZERO LEAKAGE: FAIL  ({n_viol} overlapping dialogue_ids)')

    leakage_results.append({'fold': fold_id, 'status': status, 'violations': n_viol})

    assert n_viol == 0, f'Fold {fold_id} has {n_viol} train/val dialogue overlap violations!'

print()
all_outer_pass = all(r['violations'] == 0 for r in leakage_results)
print(f'ALL OUTER FOLDS: ZERO LEAKAGE: {"PASS" if all_outer_pass else "FAIL"}')

In [ ]:
# ---------------------------------------------------------------------------
# 3.2  Zero-leakage test — inner (nested) folds  (20 × inner CV)
# ---------------------------------------------------------------------------
print('=' * 60)
print('ZERO-LEAKAGE UNIT TESTS — INNER NESTED FOLDS')
print('=' * 60)

inner_sgkf = StratifiedGroupKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEEDS[0])

inner_pass_count = 0
inner_fail_count = 0

for outer_fold in range(cfg.N_FOLDS):
    outer_train_mask = fold_df['fold_id'] != outer_fold
    inner_df = fold_df[outer_train_mask].copy()

    X_inner = np.arange(len(inner_df))
    y_inner = inner_df['label'].values
    g_inner = inner_df['dialogue_id'].values

    for inner_fold, (tr_idx, va_idx) in enumerate(
        inner_sgkf.split(X_inner, y_inner, g_inner)
    ):
        inner_train_dlg = set(g_inner[tr_idx])
        inner_val_dlg   = set(g_inner[va_idx])
        overlap = inner_train_dlg & inner_val_dlg
        n_viol  = len(overlap)

        if n_viol == 0:
            inner_pass_count += 1
        else:
            inner_fail_count += 1
            print(f'  Outer {outer_fold} / Inner {inner_fold}: FAIL ({n_viol} violations)')

total_inner = inner_pass_count + inner_fail_count
print(f'Inner folds checked: {total_inner}  |  PASS: {inner_pass_count}  |  FAIL: {inner_fail_count}')
inner_all_pass = inner_fail_count == 0
print(f'ALL INNER FOLDS: ZERO LEAKAGE: {"PASS" if inner_all_pass else "FAIL"}')

# Store for final checklist
LEAKAGE_STATUS = 'PASS' if (all_outer_pass and inner_all_pass) else 'FAIL'

In [ ]:
# ---------------------------------------------------------------------------
# 3.3  Causal integrity check
# ---------------------------------------------------------------------------
# For each sample at seeker_turn_idx t, lag-k trajectory features should be
# derived from seeker_turn_idx t-k within the same dialogue.
# We verify this by checking that the fold_df is sorted consistently and
# that for each dialogue, seeker_turn_idx values are strictly monotone.
print('=' * 60)
print('CAUSAL INTEGRITY CHECK')
print('=' * 60)

causal_violations = 0

for dlg_id, grp in fold_df.groupby('dialogue_id'):
    grp_sorted = grp.sort_values('seeker_turn_idx')
    t_vals = grp_sorted['seeker_turn_idx'].values

    # Check monotone (non-decreasing) seeker_turn_idx
    if not np.all(np.diff(t_vals) >= 0):
        causal_violations += 1
        print(f'  Dialogue {dlg_id}: seeker_turn_idx not monotone: {t_vals}')
        continue

    # Check that lag-1 alignment is correct:
    # for each sample at turn t, the previous sample within the dialogue
    # should have seeker_turn_idx == t - 1 (or be the first turn, t==0)
    for k in range(1, len(t_vals)):
        expected_prev = t_vals[k] - 1
        actual_prev   = t_vals[k - 1]
        # Allow gap if dialogue has non-consecutive seeker turns (supporter-only turns)
        # We just verify actual_prev < current, which guarantees no future leakage
        if actual_prev >= t_vals[k]:
            causal_violations += 1
            print(f'  Dialogue {dlg_id}: turn {k} seeker_turn_idx={t_vals[k]} '
                  f'preceded by {actual_prev} (not strictly earlier)')

CAUSAL_STATUS = 'PASS' if causal_violations == 0 else f'FAIL ({causal_violations} violations)'
print(f'\nCAUSAL INTEGRITY: {CAUSAL_STATUS}')

---
## Section 4 — OOF Evaluation (all 4 ablation conditions)

In [ ]:
# ---------------------------------------------------------------------------
# 4.1  OOF metrics helper
# ---------------------------------------------------------------------------
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: List[str],
) -> dict:
    """Compute macro-F1 (1-8), macro-F1 (0-8), per-class F1/P/R."""
    classes_1_8 = list(range(1, 9))
    classes_0_8 = list(range(9))

    macro_f1_1_8 = f1_score(
        y_true, y_pred, labels=classes_1_8,
        average='macro', zero_division=0
    )
    macro_f1_0_8 = f1_score(
        y_true, y_pred, labels=classes_0_8,
        average='macro', zero_division=0
    )

    per_f1 = f1_score(y_true, y_pred, labels=classes_0_8,
                      average=None, zero_division=0)
    per_pr = precision_score(y_true, y_pred, labels=classes_0_8,
                             average=None, zero_division=0)
    per_rc = recall_score(y_true, y_pred, labels=classes_0_8,
                          average=None, zero_division=0)

    return {
        'macro_f1_1_8': macro_f1_1_8,
        'macro_f1_0_8': macro_f1_0_8,
        'per_f1': per_f1,
        'per_precision': per_pr,
        'per_recall': per_rc,
    }


print('compute_metrics() defined')

In [ ]:
# ---------------------------------------------------------------------------
# 4.2  Run OOF evaluation for all conditions
# ---------------------------------------------------------------------------
print('=' * 60)
print('OOF EVALUATION RESULTS')
print('=' * 60)

oof_metrics_all = []

for cond in conditions:
    m = compute_metrics(train_labels, cond['oof_preds'], cfg.CLASS_NAMES)
    cond['oof_metrics'] = m
    oof_metrics_all.append({
        'condition'     : cond['name'],
        'macro_f1_1_8'  : round(m['macro_f1_1_8'], 4),
        'macro_f1_0_8'  : round(m['macro_f1_0_8'], 4),
    })
    print(f"\n[{cond['name']}]")
    print(f"  Macro-F1 (1-8): {m['macro_f1_1_8']:.4f}")
    print(f"  Macro-F1 (0-8): {m['macro_f1_0_8']:.4f}")
    print(f"  Per-class F1  : {np.round(m['per_f1'], 3)}")

    # Per-class table
    pc_df = pd.DataFrame({
        'class': cfg.CLASS_NAMES,
        'F1'   : np.round(m['per_f1'],        3),
        'P'    : np.round(m['per_precision'],  3),
        'R'    : np.round(m['per_recall'],     3),
    })
    print(pc_df.to_string(index=False))

ablation_summary_oof = pd.DataFrame(oof_metrics_all)
print('\n')
print('=' * 60)
print('ABLATION SUMMARY (OOF)')
print('=' * 60)
print(ablation_summary_oof.to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# 4.3  Normalized OOF confusion matrices (computed here, plotted in Section 6)
# ---------------------------------------------------------------------------
for cond in conditions:
    cm = confusion_matrix(train_labels, cond['oof_preds'], labels=list(range(9)))
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-12)
    cond['oof_cm']      = cm
    cond['oof_cm_norm'] = cm_norm

print('OOF confusion matrices computed.')

---
## Section 5 — Held-Out Test Set Evaluation (ONE TIME — FINAL)

In [ ]:
# ---------------------------------------------------------------------------
# 5.1  Test-set evaluation for all conditions
# ---------------------------------------------------------------------------
print('=' * 60)
print('HELD-OUT TEST SET EVALUATION (FINAL — ONE TIME)')
print('=' * 60)

test_summary_rows = []
per_class_rows    = []

for cond in conditions:
    m = compute_metrics(test_labels, cond['test_preds'], cfg.CLASS_NAMES)
    cond['test_metrics'] = m

    print(f"\n[{cond['name']}]")
    print(f"  Macro-F1 (1-8): {m['macro_f1_1_8']:.4f}")
    print(f"  Macro-F1 (0-8): {m['macro_f1_0_8']:.4f}")

    test_summary_rows.append({
        'condition'    : cond['name'],
        'macro_f1_1_8' : round(m['macro_f1_1_8'], 4),
        'macro_f1_0_8' : round(m['macro_f1_0_8'], 4),
    })

    for cls_idx, cls_name in enumerate(cfg.CLASS_NAMES):
        per_class_rows.append({
            'condition' : cond['name'],
            'class_idx' : cls_idx,
            'class_name': cls_name,
            'F1'        : round(m['per_f1'][cls_idx], 4),
            'precision' : round(m['per_precision'][cls_idx], 4),
            'recall'    : round(m['per_recall'][cls_idx], 4),
        })

test_summary_df  = pd.DataFrame(test_summary_rows)
per_class_df     = pd.DataFrame(per_class_rows)

print('\nTest Summary:')
print(test_summary_df.to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# 5.2  Split Inflation Measurement
#      Compare 5-fold-mean (non-routed) vs held-out-routed for Stage 1
# ---------------------------------------------------------------------------
print('=' * 60)
print('SPLIT INFLATION MEASUREMENT')
print('=' * 60)

# Non-routed (naive mean of all 5 folds) for Stage 1
s1_mean_probs = softmax(s1_test_mean, axis=1)
s1_mean_preds = s1_mean_probs.argmax(axis=1)

f1_nonrouted = f1_score(test_labels, s1_mean_preds, labels=list(range(1, 9)),
                        average='macro', zero_division=0)
f1_routed    = conditions[0]['test_metrics']['macro_f1_1_8']  # Baseline (routed)

inflation = f1_nonrouted - f1_routed

print(f'Stage 1 — Mean-of-folds (non-routed) Macro-F1 (1-8): {f1_nonrouted:.4f}')
print(f'Stage 1 — Held-out-routed             Macro-F1 (1-8): {f1_routed:.4f}')
print(f'Inflation delta (non-routed - routed) : {inflation:+.4f}')

if inflation > 0:
    print('  → Non-routed inflates the score (optimistic bias from fold overlap)')
elif inflation < 0:
    print('  → Routed is higher (possibly routing picks better-fit fold)')
else:
    print('  → No inflation detected')

In [ ]:
# ---------------------------------------------------------------------------
# 5.3  Save ablation_summary.csv and per_class_metrics.csv
# ---------------------------------------------------------------------------
ablation_summary_path = RESULTS / 'ablation_summary.csv'
per_class_path        = RESULTS / 'per_class_metrics.csv'

test_summary_df.to_csv(ablation_summary_path, index=False)
per_class_df.to_csv(per_class_path, index=False)

print(f'Saved: {ablation_summary_path}')
print(f'Saved: {per_class_path}')

---
## Section 6 — Error Analysis & Confusion Matrices

In [ ]:
# ---------------------------------------------------------------------------
# 6.1  4-Panel normalized OOF confusion matrices
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
axes = axes.flatten()

for ax, cond in zip(axes, conditions):
    sns.heatmap(
        cond['oof_cm_norm'],
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=cfg.CLASS_NAMES,
        yticklabels=cfg.CLASS_NAMES,
        vmin=0, vmax=1,
        ax=ax,
        linewidths=0.3,
        cbar_kws={'shrink': 0.8},
    )
    ax.set_title(cond['name'], fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('True', fontsize=9)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.suptitle('Normalized OOF Confusion Matrices — Ablation Conditions', fontsize=14, y=1.01)
plt.tight_layout()

cm_path = RESULTS / 'confusion_matrices.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {cm_path}')

In [ ]:
# ---------------------------------------------------------------------------
# 6.2  C5 / C6 / C7 Confusion sub-matrix (Stage 1 Baseline vs Full Pipeline)
# ---------------------------------------------------------------------------
focus_classes = [5, 6, 7]  # Neurotic, Obsessional, High-Adaptive
focus_names   = [cfg.CLASS_NAMES[c] for c in focus_classes]

print('=' * 60)
print('C5/C6/C7 CONFUSION SUB-MATRIX — Stage 1 Baseline vs Full Pipeline')
print('=' * 60)

for cond_idx in [0, 3]:  # Baseline, Full Pipeline
    cond = conditions[cond_idx]
    preds = cond['test_preds']
    true  = test_labels

    # Filter to samples whose TRUE label is in focus_classes
    focus_mask = np.isin(true, focus_classes)
    sub_true   = true[focus_mask]
    sub_pred   = preds[focus_mask]

    sub_cm = confusion_matrix(sub_true, sub_pred, labels=focus_classes)
    sub_cm_norm = sub_cm.astype(float) / (sub_cm.sum(axis=1, keepdims=True) + 1e-12)

    sub_df = pd.DataFrame(sub_cm_norm, index=focus_names, columns=focus_names)
    print(f"\n[{cond['name']}] — C5/C6/C7 sub-matrix (normalized, true_label ∈ {{C5,C6,C7}})")
    print(sub_df.round(3).to_string())

    fig_sub, ax_sub = plt.subplots(figsize=(6, 4))
    sns.heatmap(
        sub_df, annot=True, fmt='.3f', cmap='Oranges',
        vmin=0, vmax=1, ax=ax_sub, linewidths=0.5
    )
    ax_sub.set_title(f"{cond['name']}\nC5/C6/C7 Confusion", fontsize=10)
    ax_sub.set_xlabel('Predicted', fontsize=9)
    ax_sub.set_ylabel('True', fontsize=9)
    plt.tight_layout()
    sub_path = RESULTS / f'cm_c567_{cond_idx}.png'
    plt.savefig(sub_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {sub_path}')

In [ ]:
# ---------------------------------------------------------------------------
# 6.3  Minority class recovery — F1 for C2, C5, C8 across all conditions
# ---------------------------------------------------------------------------
minority_classes = [2, 5, 8]  # MajorImgDist, Neurotic, NMI
minority_names   = [cfg.CLASS_NAMES[c] for c in minority_classes]

print('=' * 60)
print('MINORITY CLASS F1 RECOVERY (TEST SET)')
print('=' * 60)

minority_rows = []
for cond in conditions:
    pf1 = cond['test_metrics']['per_f1']
    row = {'condition': cond['name']}
    for c, cname in zip(minority_classes, minority_names):
        row[cname] = round(pf1[c], 4)
    minority_rows.append(row)

minority_df = pd.DataFrame(minority_rows)
print(minority_df.to_string(index=False))

# Also for OOF
print('\nMINORITY CLASS F1 RECOVERY (OOF)')
minority_oof_rows = []
for cond in conditions:
    pf1 = cond['oof_metrics']['per_f1']
    row = {'condition': cond['name']}
    for c, cname in zip(minority_classes, minority_names):
        row[cname] = round(pf1[c], 4)
    minority_oof_rows.append(row)

minority_oof_df = pd.DataFrame(minority_oof_rows)
print(minority_oof_df.to_string(index=False))

---
## Section 7 — Bootstrap Confidence Intervals

In [ ]:
# ---------------------------------------------------------------------------
# 7.1  Bootstrap CI computation
# ---------------------------------------------------------------------------
N_BOOTSTRAP = 1000
CI_ALPHA    = 0.05   # 95% CI
RNG         = np.random.default_rng(cfg.SEEDS[0])

print('=' * 60)
print(f'BOOTSTRAP 95% CI — {N_BOOTSTRAP} iterations on held-out test set')
print('=' * 60)

bootstrap_rows = []

for cond in conditions:
    preds = cond['test_preds']
    true  = test_labels
    n     = len(true)

    boot_scores = []
    for _ in range(N_BOOTSTRAP):
        idx = RNG.integers(0, n, size=n)
        score = f1_score(
            true[idx], preds[idx],
            labels=list(range(1, 9)),
            average='macro',
            zero_division=0
        )
        boot_scores.append(score)

    boot_scores = np.array(boot_scores)
    mean_score  = boot_scores.mean()
    ci_lower    = np.percentile(boot_scores, 2.5)
    ci_upper    = np.percentile(boot_scores, 97.5)

    cond['ci_mean']  = mean_score
    cond['ci_lower'] = ci_lower
    cond['ci_upper'] = ci_upper

    print(f"[{cond['name']}]")
    print(f"  Bootstrap Mean: {mean_score:.4f}  CI: [{ci_lower:.4f}, {ci_upper:.4f}]")

    bootstrap_rows.append({
        'condition'    : cond['name'],
        'macro_f1_1_8' : round(cond['test_metrics']['macro_f1_1_8'], 4),
        'macro_f1_0_8' : round(cond['test_metrics']['macro_f1_0_8'], 4),
        'bs_mean'      : round(mean_score, 4),
        'ci_lower'     : round(ci_lower,   4),
        'ci_upper'     : round(ci_upper,   4),
    })

boot_df = pd.DataFrame(bootstrap_rows)
print('\nFull table:')
print(boot_df.to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# 7.2  Save final ablation_summary.csv with CI columns
# ---------------------------------------------------------------------------
boot_df.to_csv(ablation_summary_path, index=False)
print(f'Updated: {ablation_summary_path} (with ci_lower, ci_upper)')

# Forest-plot style bar chart with CI
fig, ax = plt.subplots(figsize=(10, 5))
y_pos   = np.arange(len(conditions))
colors  = ['#4c78a8', '#54a24b', '#e45756', '#9467bd']

for i, cond in enumerate(conditions):
    ax.barh(
        y_pos[i],
        cond['test_metrics']['macro_f1_1_8'],
        xerr=[
            [cond['test_metrics']['macro_f1_1_8'] - cond['ci_lower']],
            [cond['ci_upper'] - cond['test_metrics']['macro_f1_1_8']]
        ],
        color=colors[i], alpha=0.8, capsize=5, height=0.5,
        error_kw={'elinewidth': 1.5, 'capthick': 1.5}
    )
    ax.text(
        cond['test_metrics']['macro_f1_1_8'] + 0.002,
        y_pos[i],
        f"{cond['test_metrics']['macro_f1_1_8']:.4f}",
        va='center', fontsize=9
    )

ax.set_yticks(y_pos)
ax.set_yticklabels([c['name'] for c in conditions], fontsize=9)
ax.set_xlabel('Macro-F1 (classes 1–8)', fontsize=10)
ax.set_title('Ablation Study — Held-Out Test Macro-F1 with 95% Bootstrap CI', fontsize=11)
ax.axvline(conditions[0]['test_metrics']['macro_f1_1_8'], color='gray', linestyle='--', alpha=0.5, label='Baseline')
ax.legend(fontsize=8)
plt.tight_layout()

ci_plot_path = RESULTS / 'ablation_ci_plot.png'
plt.savefig(ci_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {ci_plot_path}')

---
## Section 8 — Cross-Seed Variance

In [ ]:
# ---------------------------------------------------------------------------
# 8.1  Multi-seed variance (if available)
# ---------------------------------------------------------------------------
print('=' * 60)
print('CROSS-SEED VARIANCE CHECK')
print('=' * 60)

seed_results = []

for seed in cfg.SEEDS:
    seed_oof_path = ART_DIR / 'stage1_qwen3_8b' / f'seed_{seed}' / 'oof_logits.npy'
    if seed_oof_path.exists():
        oof = np.load(seed_oof_path)
        preds = softmax(oof, axis=1).argmax(axis=1)
        f1 = f1_score(
            train_labels, preds,
            labels=list(range(1, 9)),
            average='macro', zero_division=0
        )
        seed_results.append({'seed': seed, 'oof_macro_f1_1_8': round(f1, 4)})
        print(f'  Seed {seed:>4}: OOF Macro-F1 (1-8) = {f1:.4f}')
    else:
        print(f'  Seed {seed:>4}: NOT FOUND — {seed_oof_path}')

if len(seed_results) > 1:
    scores = np.array([r['oof_macro_f1_1_8'] for r in seed_results])
    print(f'\nCross-seed mean  : {scores.mean():.4f}')
    print(f'Cross-seed std   : {scores.std():.4f}')
    print(f'Cross-seed range : [{scores.min():.4f}, {scores.max():.4f}]')

    seed_df = pd.DataFrame(seed_results)
    seed_df.to_csv(RESULTS / 'cross_seed_variance.csv', index=False)
    print(f'Saved: {RESULTS / "cross_seed_variance.csv"}')
elif len(seed_results) == 1:
    print('\nNOTE: Only seed_42 results are available.')
    print('Single-seed results are reported. For robustness, re-run training with:')
    print('  Seeds: 2026 and 777')
    print('  Save OOF logits to:')
    print(f'    {ART_DIR}/stage1_qwen3_8b/seed_2026/oof_logits.npy')
    print(f'    {ART_DIR}/stage1_qwen3_8b/seed_777/oof_logits.npy')
else:
    print('\nNOTE: No seed results found.')
    print('Run training notebooks (02 & 03) first to generate oof_logits.npy.')

---
## Section 9 — Final Reproducibility Check

In [ ]:
# ---------------------------------------------------------------------------
# 9.1  Reproducibility checklist
# ---------------------------------------------------------------------------
print('=' * 60)
print('FINAL REPRODUCIBILITY CHECKLIST')
print('=' * 60)

def check(path):
    return '[x]' if Path(path).exists() else '[ ]'

checklist = [
    (check(RESULTS / 'ablation_summary.csv'),
     'ablation_summary.csv written'),
    (check(RESULTS / 'per_class_metrics.csv'),
     'per_class_metrics.csv written'),
    (check(RESULTS / 'confusion_matrices.png'),
     'confusion_matrices.png written'),
    (check(RESULTS / 'ablation_ci_plot.png'),
     'ablation_ci_plot.png written (bootstrap CI)'),
    (check(RESULTS / 'cm_c567_0.png') and check(RESULTS / 'cm_c567_3.png'),  # noqa: E501
     'C5/C6/C7 sub-matrices written'),
    ('[x]' if LEAKAGE_STATUS == 'PASS' else f'[!] FAIL',
     f'Zero leakage test: {LEAKAGE_STATUS}'),
    ('[x]' if 'PASS' in CAUSAL_STATUS else '[!]',
     f'Causal integrity test: {CAUSAL_STATUS}'),
]

for status, desc in checklist:
    print(f'  {status}  {desc}')

all_check = all(s.startswith('[x]') for s, _ in checklist)
print()
if all_check:
    print('✅ ALL CHECKS PASSED — notebook is reproducibility-complete.')
else:
    print('⚠️  Some checks FAILED — review items marked [!] or [ ] above.')

In [ ]:
# ---------------------------------------------------------------------------
# 9.2  Final summary printout
# ---------------------------------------------------------------------------
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)
print()
print('OOF Ablation Summary:')
print(ablation_summary_oof.to_string(index=False))
print()
print('Test Set Results with Bootstrap CI:')
print(boot_df[['condition', 'macro_f1_1_8', 'ci_lower', 'ci_upper']].to_string(index=False))
print()
print('Minority Class F1 on Test Set:')
print(minority_df.to_string(index=False))
print()
print('All outputs saved to:', str(RESULTS))